# Cross-y analysis

In previous experiments, the fitted u only correspond to a single observation y. This notebook now tries to analyse how a certain feature could affect several observations simultaneously.

In [1]:
import sys
import os
import pickle

sys.path.insert(0, os.path.abspath(".."))

import numpy as np

from sklearn.preprocessing import StandardScaler

from src.function_library import build_function_library,build_interaction_library, select_top_power_features, power_features, build_power_library
from src.evaluation import build_theta_from_model, visualize_sparse_prediction, analyze_feature_cmi, analyze_feature_importance
from src.sparse_interp import sparse_ee_interpretation, save_sparse_result, load_sparse_result, sparse_predict, refine_sparse_result

from npeet import entropy_estimators as ee
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from itertools import combinations

In [2]:
def ee_objective(coeffs, X, y):
    coeffs = np.asarray(coeffs, dtype=float)
    coeffs /= np.linalg.norm(coeffs)
    u = X @ coeffs
    return ee.mi(y, u)

def scipy_objective(coeffs, X, y):
    return -ee_objective(coeffs, X, y)

In [3]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\final_data.npz")
print(data.files)

X_final = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_final = data["v_lv"]
v_rv_final = data["v_rv"]

y_v_lv_max_final = np.max(v_lv_final, axis=1)
y_v_lv_min_final = np.min(v_lv_final, axis=1)

y_v_rv_max_final = np.max(v_rv_final, axis=1)
y_v_rv_min_final = np.min(v_rv_final, axis=1)

y_v_lv_mean_final = np.mean(v_lv_final, axis=1)
y_v_rv_mean_final = np.mean(v_rv_final, axis=1)

y_v_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_lv_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_lv_min_final,
    y_v_lv_mean_final,
])
y_v_rv_combined = np.column_stack([
    y_v_rv_max_final,
    y_v_rv_min_final,
    y_v_rv_mean_final,
])

y_v_max_combined = np.column_stack([
    y_v_lv_max_final,
    y_v_rv_max_final,
])
y_v_min_combined = np.column_stack([
    y_v_lv_min_final,
    y_v_rv_min_final,
])
y_v_mean_combined = np.column_stack([
    y_v_lv_mean_final,
    y_v_rv_mean_final,
])


['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


In [4]:
# load old sampling
with open("../notebook/final_data_split.pkl", "rb") as f:
    final_data_split = pickle.load(f)

final_train_idx = final_data_split["train_idx"]
final_test_idx = final_data_split["test_idx"]

X_final_train = X_final[final_train_idx]
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]

y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]

y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]

y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]

y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

# LV combined
y_v_lv_combined_train = y_v_lv_combined[final_train_idx]
y_v_lv_combined_test = y_v_lv_combined[final_test_idx]

# RV combined
y_v_rv_combined_train = y_v_rv_combined[final_train_idx]
y_v_rv_combined_test = y_v_rv_combined[final_test_idx]

# Max combined
y_v_max_combined_train = y_v_max_combined[final_train_idx]
y_v_max_combined_test = y_v_max_combined[final_test_idx]

# Min combined
y_v_min_combined_train = y_v_min_combined[final_train_idx]
y_v_min_combined_test = y_v_min_combined[final_test_idx]

# Mean combined
y_v_mean_combined_train = y_v_mean_combined[final_train_idx]
y_v_mean_combined_test = y_v_mean_combined[final_test_idx]

# All combined
y_v_combined_train = y_v_combined[final_train_idx]
y_v_combined_test = y_v_combined[final_test_idx]

In [8]:
# original library
# Theta_v, feature_names_v = build_function_library(
#     X_final_train,
#     param_names
# )

Theta_v, feature_names_v = (
    build_interaction_library(
        X_final_train,
        param_names
    )
)

In [13]:
def analyze_joint_mi(
    Theta,
    feature_names,
    Y,
    k=1,
):
    Theta = np.asarray(Theta)
    Y = np.asarray(Y)

    results = []

    for indices in combinations(range(Theta.shape[1]), k):

        X_combined = Theta[:, indices]

        mi = ee.mi(
            X_combined,
            Y
        )

        feature_combination = tuple(
            feature_names[i]
            for i in indices
        )

        results.append({
            "features": feature_combination,
            "mi": mi
        })

    results.sort(
        key=lambda x: x["mi"],
        reverse=True
    )

    return results

def print_joint_mi_results(
    results_dict,
    top_n=50,
):
    combinations_list = []

    for results in results_dict.values():
        for result in results:
            feature_combination = result["features"]

            if feature_combination not in combinations_list:
                combinations_list.append(feature_combination)

    # Store MI values
    mi_table = {}

    for feature_combination in combinations_list:
        mi_table[feature_combination] = {}

    for target_name, results in results_dict.items():

        for result in results:

            feature_combination = result["features"]
            mi = result["mi"]

            mi_table[feature_combination][target_name] = mi

    # Print header
    target_names = list(results_dict.keys())

    header = f"{'Features':50s}"

    for target_name in target_names:
        header += f"{target_name:>12s}"

    print(header)
    print("-" * len(header))

    # Sort by first target
    first_target = target_names[0]

    combinations_list.sort(
        key=lambda combination:
            mi_table[combination].get(
                first_target,
                float("-inf")
            ),
        reverse=True
    )

    # Only display top N
    combinations_to_print = combinations_list[:top_n]

    # Print rows
    for feature_combination in combinations_to_print:

        feature_string = " & ".join(
            feature_combination
        )

        row = f"{feature_string:50s}"

        for target_name in target_names:

            mi = mi_table[feature_combination].get(
                target_name,
                float("nan")
            )

            row += f"{mi:12.6f}"

        print(row)

    return mi_table

def save_joint_mi_results(
    results,
    filename,
):
    with open(filename, "wb") as f:
        pickle.dump(results, f)


def load_joint_mi_results(
    filename,
):
    with open(filename, "rb") as f:
        results = pickle.load(f)

    return results

In [10]:
results_v_lv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_lv_combined_train,
    k=2
)
results_v_rv = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_rv_combined_train,
    k=2
)
results_v_max = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_max_combined_train,
    k=2
)
results_v_min = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_min_combined_train,
    k=2
)
results_v_mean = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_mean_combined_train,
    k=2
)
results_v = analyze_joint_mi(
    Theta_v,
    feature_names_v,
    y_v_combined_train,
    k=2
)

In [14]:
results_dict = {
    "All": results_v,
    "LV": results_v_lv,
    "RV": results_v_rv,
    "Max": results_v_max,
    "Min": results_v_min,
    "Mean": results_v_mean,
}

save_joint_mi_results(
    results_dict,
    "../results/joint_mi_k4.pkl"
)

In [15]:
results_dict = load_joint_mi_results(
    "../results/joint_mi_k4.pkl"
)

mi_table = print_joint_mi_results(
    results_dict
)

Features                                                   All          LV          RV         Max         Min        Mean
--------------------------------------------------------------------------------------------------------------------------
R_p & Emax_lv/Emin_lv                                 1.085470    0.989414    0.715104    0.851802    0.731803    0.694974
R_p & Emax_lv/Emax_rv                                 1.050589    1.069456    1.047064    0.845082    0.643412    0.728844
R_s & Emax_lv/Emax_rv                                 1.031122    0.785203    0.856175    0.866093    0.646188    0.668895
R_s & Emin_lv/Emin_rv                                 1.016936    0.758028    0.848853    0.856503    0.622844    0.658380
R_s & Emax_lv+Emin_rv                                 1.014085    0.763426    0.850040    0.850787    0.628070    0.662647
R_s & Emax_lv-Emin_lv                                 1.013566    0.763640    0.846679    0.849500    0.628706    0.667777
R_s & Emax_lv-Em